# Подбор порога для неполных дублей FAQ по E5

Ноутбук получает соседей по cosine готовых эмбеддингов ответов. После поиска проверяет контекст пары: продукт, операцию, числа, запреты и ссылки. Полные совпадения попадают на отдельную вкладку. Итог — кандидаты для редакторской проверки.

**Вход** — две выгрузки FAQ, полный `faq_embeddings_final.npy` и снимок ID и ответов в порядке строк того же запуска. Укажите пути ниже и выберите **Run All**. Для выполнения нужны Python 3.10+, `numpy`, `pandas` и `openpyxl`.

**Для автоматического подбора порога** положите локальный файл `faq_e5_audit.json` рядом с ноутбуком или укажите путь к своей разметке. Файл аудита содержит отпечатки карточек и метки пар.

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
EXPORT_FILES = [
    ROOT / "data" / "export_faq_idp_main_elch.xlsx",
    ROOT / "data" / "export_faq_idp_additional_elch.xlsx",
]
E5_NPY = ROOT / "faq_embeddings_final.npy"
# Доверенный локальный PKL или CSV со столбцами «Идентификатор» и «Ответ».
E5_SNAPSHOT = ROOT / "faq_answers_snapshot.pkl"
AUDIT_FILE = ROOT / "faq_e5_audit.json"
BENCHMARK_FILE = ROOT / "faq_partial_benchmark.csv"
if not BENCHMARK_FILE.is_file():
    BENCHMARK_FILE = ROOT / "results" / "faq_partial_benchmark.csv"
OUTPUT_DIR = ROOT / "results" / "e5_run"

TOP_K = 12
REQUIRE_FULL_COVERAGE = False

## Проверка и результаты

В поиск попадают карточки, у которых совпали ID и текст ответа в кэше и в текущей выгрузке. При частичном покрытии выводится число карточек без вектора. Косинус рассчитывается блоками из уже сохранённых векторов. `TOP_K` действует отдельно для соседей из того же продукта и из других продуктов.

Ноутбук сопоставляет ранее вычитанные пары по отпечаткам карточек. Сравнивает cosine E5, проверки условий и добавление направленной вложенности. На части «подбор» выбираются порог и правило по F1. Часть «проверка» используется для оценки выбранного порога. Неоднозначные метки исключаются. В варианте с вложенностью паре со строгой текстовой вложенностью или высоким направленным покрытием разрешается не дотянуть до базового порога на 0,02.

В первом запуске достаточно вывода последней ячейки. Для сохранения HTML и Excel в `OUTPUT_DIR` включите `WRITE_REPORT = True`. Метрики относятся к размеченной выборке. Для оценки работы RAG нужна проверка на реальных поисковых запросах.

Для обсуждения порога можно сфотографировать вывод двух последних ячеек. Таблица «Объём по порогам» показывает, сколько пар проходит каждый порог до проверки условий. Количество пар само по себе не показывает качество. В отчёте с разметкой сравнивайте число найденных полезных пар и число лишних.

In [ ]:
from dataclasses import dataclass
from collections import Counter
import re
from openpyxl import load_workbook

CONTEXT_FIELDS = ('Продукт', 'Подпродукт', 'Тематика', 'Основной канал',
                  'Дополнительный канал', 'Дата, время включения', 'Дата, время отключения')
MAIN, EXTRA = 'Основной канал', 'Дополнительный канал'
LEGACY = 'Перенесено из предыдущего отчёта'
SAME, DIFFERENT, UNKNOWN = 'same', 'different', 'unknown'
INSURANCE_PROGRAM = re.compile(r'\bзащит[аеуы]\s+(дома|денег)\b', re.I)
DEPOSIT_CLOSE = re.compile(r'^как\s+(?:досрочно\s+)?закрыть\s+вклад\s+[«"]([^»"]+)[»"]\s*\??$', re.I)
NUMBERS = re.compile(r'\d{1,3}(?:[ \u00a0]\d{3})+(?:[.,]\d+)?|\d+(?:[.,:/-]\d+)*')
NEGATION = re.compile(r'\b(?:не|нет|нельзя|без)\b', re.I)


def clean(value):
    return '' if value is None else str(value)


def norm(value):
    return ' '.join(clean(value).split())


def numbers(answer):
    return {re.sub(r'\s', '', token).replace(',', '.')
            for token in NUMBERS.findall(answer)}


def differences(a, b):
    links = {field for field in a.row.keys() | b.row.keys() if field and field.startswith('Ссылка')}
    fields = [field for field in (*CONTEXT_FIELDS, *sorted(links))
              if norm(a.row.get(field)) != norm(b.row.get(field))]
    if numbers(a.answer) != numbers(b.answer):
        fields.append('Числа в тексте')
    if Counter(NEGATION.findall(a.answer.casefold())) != Counter(NEGATION.findall(b.answer.casefold())):
        fields.append('Отрицания в тексте')
    return fields


@dataclass(frozen=True)
class FAQ:
    id: str
    question: str
    answer: str
    alternatives: tuple
    source: str
    row: dict

    @property
    def product_key(self):
        return tuple(norm(self.row.get(field)).casefold() for field in CONTEXT_FIELDS[:2])


def context(a, b):
    ap, asub = a.product_key
    bp, bsub = b.product_key
    if ap and bp and ap != bp:
        return DIFFERENT, 'Разные продукты'
    if not ap or not bp or not asub or not bsub:
        return UNKNOWN, 'У одной или обеих карточек не заполнен продукт или подпродукт'
    if asub != bsub:
        return DIFFERENT, 'Разные подпродукты'
    program = None
    if (ap, asub) == ('страхование', 'злс'):
        program = INSURANCE_PROGRAM
    elif ap == 'вклады и счета':
        program = DEPOSIT_CLOSE
    if program:
        left, right = (program.search(norm(card.question)) for card in (a, b))
        if left and right and norm(left[1]).casefold() != norm(right[1]).casefold():
            return DIFFERENT, 'Разные программы внутри подпродукта'
    return SAME, 'Совпадают продукт и подпродукт'


def read_records(paths):
    records, seen = {}, set()
    counts = Counter()
    required = {'Идентификатор', 'Вопрос', 'Ответ', 'Продукт', 'Подпродукт', 'Статус', MAIN, EXTRA}
    for path in paths:
        book = load_workbook(path, read_only=True, data_only=True)
        try:
            rows = iter(book['Выгрузка'].values)
            fields = next(rows)
            if missing := required.difference(fields):
                raise ValueError(f'{path.name}: отсутствуют поля {sorted(missing)}')
            alt_fields = [f for f in fields if f and f.startswith('Альтернативный вопрос')]
            for line, values in enumerate(rows, 2):
                if all(value is None for value in values):
                    continue
                counts['Строк в выгрузках'] += 1
                row = dict(zip(fields, values))
                ident = norm(row['Идентификатор'])
                if not ident or ident in seen:
                    raise ValueError(f'{path.name}:{line}: пустой или повторяющийся ID FAQ')
                seen.add(ident)
                main, extra = ('ЕЛЧ' in {norm(x) for x in clean(row[key]).split(',')}
                               for key in (MAIN, EXTRA))
                if norm(row['Статус']) != 'Активно' or not (main or extra):
                    continue
                if not norm(row['Вопрос']) or not norm(row['Ответ']):
                    raise ValueError(f'{path.name}:{line}: пустой вопрос или ответ')
                source = 'main' if main else 'additional'
                counts[source] += 1
                records[ident] = FAQ(ident, clean(row['Вопрос']), clean(row['Ответ']),
                                     tuple(clean(row[f]) for f in alt_fields if norm(row[f])),
                                     source, row)
        finally:
            book.close()
    return records, counts

In [ ]:
"""Lightweight, offline candidate search for incomplete FAQ duplicates."""

from collections import Counter, defaultdict
from dataclasses import dataclass
import math
from pathlib import Path
import re
import unicodedata

import numpy as np


TOKEN = re.compile(r'[^\W\d_]+|\d+(?:[.,:/-]\d+)*', re.UNICODE)
LIST_NUMBER = re.compile(r'(?m)^\s*\d{1,2}[.)]\s+')
RISK_NUMBER = re.compile(
    r'(?<!\w)(\d+(?:[ \u00a0]\d{3})*(?:[.,]\d+)?)\s*'
    r'(%|рабоч\w*\s+(?:дн\w*|день)|календар\w*\s+(?:дн\w*|день)|дн(?:я|ей)?\b|день\b|'
    r'руб\w*|₽|тыс\w*|млн\w*|мес\w*|год\w*|лет\b|час\w*)', re.I)
ARTICLE = re.compile(r'\b(статья|пункт|подпункт)\s*(\d+(?:\.\d+)*)', re.I)
QUOTES = re.compile(r'[«"]([^»"]+)[»"]')
NAMED_DEPOSIT = re.compile(r'\b(?:вклад|сч[её]т)\w*\s*[«"]([^»"]+)[»"]', re.I)
PRIME = re.compile(r'\bсберпрайм\b(?:\s*(?:\+|плюс))?', re.I)
SCOPE = re.compile(r'\b(?:сберперв\w*|сберпремьер\w*|доступн\w* выгод\w*|свой максимум|'
                   r'подушка безопасности|страхование по часам|дмс|защита от травм)\b')
URL = re.compile(r'https?://\S+', re.I)
QUOTE_CHARS = str.maketrans('', '', '«»„“”"')
DASH_CHARS = str.maketrans({'—': '–', '−': '–'})
OPERATIONS = [
    (r'\b(?:внос\w*|внести|полож(?!ен)\w*|пополн\w*)\b', r'\b(?:снят\w*|сним\w*|выдал|выдач\w*)\b', 'Внесение и выдача'),
    (r'\bподключ\w*\b', r'\bотключ\w*\b', 'Подключение и отключение'),
    (r'\bдобав\w*\b', r'\bудал\w*\b', 'Добавление и удаление'),
    (r'\bоткр\w*\b', r'\bзакр\w*\b', 'Открытие и закрытие'),
]
OPERATIONS = [(re.compile(a), re.compile(b), label) for a, b, label in OPERATIONS]
STRONG_NEGATION = re.compile(r'\b(?:нельзя|запрещ\w*|не\s+допуска\w*|не\s+предусмотр\w*)\b')




def words(text):
    return TOKEN.findall(LIST_NUMBER.sub('', text).casefold().replace('ё', 'е'))


def answer_key(text):
    # Preserve units, +, digits, punctuation inside numbers, and case-sensitive URLs.
    if re.search(r'https?://', text, re.I):
        return norm(text)
    normalized = unicodedata.normalize('NFC', text).translate(QUOTE_CHARS).translate(DASH_CHARS)
    return norm(normalized.casefold().replace('ё', 'е')).removesuffix('.').rstrip()


def weighted_matrix(documents, max_df=.20):
    """Return unit TF-IDF rows for unigrams and two-word phrases."""
    from scipy.sparse import csr_matrix

    count = len(documents)
    terms = [Counter([*tokens, *zip(tokens, tokens[1:])]) for tokens in documents]
    frequency = Counter(term for row in terms for term in row)
    vocab = {term: k for k, term in enumerate(
        term for term, df in frequency.items() if df <= max(2, count * max_df))}
    indptr, indices, data = [0], [], []
    for row in terms:
        weighted = [(vocab[t], (1 + math.log(n)) * (1 + math.log((1 + count) / (1 + frequency[t]))))
                    for t, n in row.items() if t in vocab]
        size = math.sqrt(sum(w * w for _, w in weighted))
        indices.extend(i for i, _ in weighted)
        data.extend(w / size for _, w in weighted)
        indptr.append(len(data))
    result = csr_matrix((np.asarray(data, dtype=np.float32), indices, indptr),
                        shape=(count, len(vocab)), dtype=np.float32)
    result.sort_indices()
    return result


def embedding_cosines(pairs, dense):
    scores = [None] * len(pairs)
    if dense is None:
        return scores
    vectors, positions = dense
    hits = [(k, positions[a], positions[b]) for k, (a, b) in enumerate(pairs)
            if a in positions and b in positions]
    if hits:
        indices, left, right = np.asarray(hits).T
        values = np.einsum('ij,ij->i', vectors[left], vectors[right]).clip(-1, 1)
        for index, value in zip(indices, values):
            scores[index] = float(value)
    return scores


class Corpus:
    def __init__(self, records, *, lexical=True):
        self.cards = [records[key] for key in sorted(records)]
        self.index = {card.id: i for i, card in enumerate(self.cards)}
        self.form = [answer_key(c.answer) for c in self.cards]
        self.answer_matrix = self.question_matrix = None
        answers = [words(c.answer) for c in self.cards]
        if lexical:
            questions = [words(c.question) for c in self.cards]
            self.answer_matrix = weighted_matrix(answers)
            self.question_matrix = weighted_matrix(questions, max_df=.40)
        counts = Counter(w for tokens in answers for w in set(tokens))
        self.idf = {w: 1 + math.log((1 + len(answers)) / (1 + n)) for w, n in counts.items()}
        self.tokens = [Counter(tokens) for tokens in answers]
        self.token_text = [' ' + ' '.join(tokens) + ' ' for tokens in answers]
        self.token_count = list(map(len, answers))
        self.weight = [sum(self.idf[w] * n for w, n in ts.items()) for ts in self.tokens]
        self.question_forms = [{norm(q).casefold() for q in (c.question, *c.alternatives)}
                               for c in self.cards]
        self.product = np.array([c.product_key[0] for c in self.cards], dtype=object)

    def pool(self, top_k=12, block=96, answer_floor=.36, question_floor=.58, dense=None,
             dense_search_floor=None):
        """Keep independent neighbour budgets inside and across products for each signal."""
        if top_k < 1 or block < 1:
            raise ValueError('Размер блока и число соседей должны быть положительными.')
        seen = defaultdict(set)
        forms = np.asarray(self.form, dtype=object)

        def select(row, columns, scores, tag):
            if not columns.size:
                return
            eligible = (columns != row) & (forms[columns] != forms[row])
            same = (self.product[columns] == self.product[row]) & bool(self.product[row])
            for mask in (eligible & same, eligible & ~same):
                offsets = np.nonzero(mask)[0]
                if offsets.size > top_k:
                    border = np.partition(scores[offsets], -top_k)[-top_k]
                    offsets = offsets[scores[offsets] >= border]
                    order = np.lexsort((columns[offsets], -scores[offsets]))[:top_k]
                    offsets = offsets[order]
                for other in columns[offsets]:
                    seen[tuple(sorted((row, int(other))))].add(tag)

        if self.answer_matrix is None and dense_search_floor is None:
            raise ValueError('Для поиска E5 нужны готовые векторы и порог поиска соседей.')
        channels = ((self.answer_matrix, answer_floor, 'ответы'),
                    (self.question_matrix, question_floor, 'вопросы'))
        for matrix, floor, tag in channels:
            if matrix is None:
                continue
            transpose = matrix.T
            for start in range(0, len(self.cards), block):
                batch = (matrix[start:start + block] @ transpose).tocsr()
                for offset in range(batch.shape[0]):
                    lo, hi = batch.indptr[offset:offset + 2]
                    columns, scores = batch.indices[lo:hi], batch.data[lo:hi]
                    good = scores >= floor
                    select(start + offset, columns[good], scores[good], tag)

        if self.question_matrix is not None:
            # Alternative-question overlaps supplement lexical neighbours only.
            inverted = defaultdict(list)
            for row, forms in enumerate(self.question_forms):
                for form in forms:
                    if len(words(form)) >= 3:
                        inverted[form].append(row)
            for rows in inverted.values():
                if len(rows) > 24:
                    continue
                for n, i in enumerate(rows):
                    for j in rows[n + 1:]:
                        if self.form[i] != self.form[j]:
                            seen[i, j].add('та же формулировка вопроса')

        if dense_search_floor is not None:
            if dense is None:
                raise ValueError('Для поиска соседей E5 нужен проверенный кэш.')
            vectors, positions = dense
            keys = np.array(sorted(positions), dtype=int)
            if keys.size:
                sample = vectors[[positions[int(i)] for i in keys]]
                for start in range(0, len(keys), block):
                    scores = sample[start:start + block] @ sample.T
                    for offset, score in enumerate(scores):
                        row = int(keys[start + offset])
                        keep = score >= dense_search_floor
                        select(row, keys[keep], score[keep], 'E5')
        return seen

    def features(self, pairs, dense=None):
        keys = sorted(pairs)
        for start in range(0, len(keys), 1024):
            batch = keys[start:start + 1024]
            if self.answer_matrix is not None:
                ai, bi = np.asarray(batch).T
                score_a = np.asarray(self.answer_matrix[ai].multiply(self.answer_matrix[bi]).sum(axis=1)).ravel()
                score_q = np.asarray(self.question_matrix[ai].multiply(self.question_matrix[bi]).sum(axis=1)).ravel()
            else:
                score_a = score_q = [None] * len(batch)
            for (i, j), a_score, q_score, dense_score in zip(batch, score_a, score_q, embedding_cosines(batch, dense)):
                a, b = self.cards[i], self.cards[j]
                shared = sum(self.idf[w] * n for w, n in (self.tokens[i] & self.tokens[j]).items())
                cov_a = shared / self.weight[i] if self.weight[i] else 0
                cov_b = shared / self.weight[j] if self.weight[j] else 0
                same_question = bool(self.question_forms[i] & self.question_forms[j])
                score = {
                    'a': a.id, 'b': b.id, 'answer': None if a_score is None else min(1., float(a_score)),
                    'question': None if q_score is None else min(1., float(q_score)), 'same_question': same_question,
                    'cover_a': min(1., cov_a), 'cover_b': min(1., cov_b), 'embedding': dense_score,
                    'literal': min(self.token_count[i], self.token_count[j]) >= 8
                    and (self.token_text[i] in self.token_text[j] or self.token_text[j] in self.token_text[i]),
                }
                yield (i, j), score


def _numbers_with_units(text):
    result = defaultdict(set)
    for value, unit in RISK_NUMBER.findall(text.casefold()):
        unit = norm(unit)
        if unit.startswith(('рабоч', 'календар')):
            unit = unit.split()[0][:5] + ' дни'
        elif unit.startswith(('дн', 'ден')):
            unit = 'дни'
        elif unit == 'лет':
            unit = 'год'
        else:
            unit = unit[:3]
        result[unit].add(float(value.replace(' ', '').replace('\u00a0', '').replace(',', '.')))
    return result


def risks(a, b):
    relation, subtype = context(a, b)
    qa, qb = [norm(x.question).casefold().replace('ё', 'е') for x in (a, b)]
    qa_refs = {(k.casefold(), n) for k, n in ARTICLE.findall(qa)}
    qb_refs = {(k.casefold(), n) for k, n in ARTICLE.findall(qb)}
    hard, caution = [], []
    same_product = bool(a.product_key[0] and a.product_key[0] == b.product_key[0])
    if relation == 'different' and 'внутри' in subtype:
        hard.append(subtype)
    elif relation != 'same':
        caution.append(subtype)
    if any({n for k2, n in qa_refs if k2 == k}.isdisjoint({n for k2, n in qb_refs if k2 == k})
           for k in set(k for k, _ in qa_refs) & set(k for k, _ in qb_refs)):
        hard.append('Разные статьи или пункты')
    names_a = {norm(s) for s in QUOTES.findall(qa)}
    names_b = {norm(s) for s in QUOTES.findall(qb)}
    if same_product and a.product_key[0] == 'вклады и счета':
        names_a = names_a or {norm(s) for s in NAMED_DEPOSIT.findall(a.answer[:160].casefold())}
        names_b = names_b or {norm(s) for s in NAMED_DEPOSIT.findall(b.answer[:160].casefold())}
    prime_a, prime_b = PRIME.search(qa), PRIME.search(qb)
    if prime_a and prime_b and (prime_a[0] == 'сберпрайм') != (prime_b[0] == 'сберпрайм'):
        hard.append('Разные пакеты СберПрайм')
    scope_a, scope_b = set(SCOPE.findall(qa)), set(SCOPE.findall(qb))
    if scope_a and scope_b and scope_a.isdisjoint(scope_b):
        caution.append('Отличается явно названная программа или пакет')
    named_scope = a.product_key[0] in ('вклады и счета', 'страхование', 'инвестиции и ценные бумаги ук')
    if same_product and named_scope and names_a and names_b and names_a.isdisjoint(names_b):
        hard.append('Разные явные названия продукта или программы')
    for x, y, reason in OPERATIONS:
        a_action, b_action = (bool(x.search(qa)), bool(y.search(qa))), (bool(x.search(qb)), bool(y.search(qb)))
        if (a_action, b_action) in (((True, False), (False, True)), ((False, True), (True, False))):
            hard.append('Разные операции: ' + reason)
    na, nb = _numbers_with_units(a.answer), _numbers_with_units(b.answer)
    for unit in sorted(na.keys() & nb.keys()):
        if na[unit].isdisjoint(nb[unit]):
            hard.append(f'Разные значения ({unit}): {sorted(na[unit])} ↔ {sorted(nb[unit])}')
        elif na[unit] != nb[unit]:
            caution.append(f'Дополнительные числовые условия ({unit})')
    if bool(STRONG_NEGATION.search(a.answer.casefold())) != bool(STRONG_NEGATION.search(b.answer.casefold())):
        caution.append('Явный запрет только в одном ответе')
    if (set(URL.findall(a.answer)) != set(URL.findall(b.answer))
            or any(flag.startswith('Ссылка') for flag in differences(a, b))):
        caution.append('Разные ссылки')
    return same_product, relation, hard, caution




def load_embedding_cache(path, snapshot, corpus, *, expected_dimension=None, require_full_coverage=False):
    """Match row positions from the original run to current cards by ID and answer text."""
    import pandas as pd

    vectors = np.load(path, mmap_mode='r', allow_pickle=False)
    if vectors.ndim != 2 or vectors.shape[1] < 1 or vectors.dtype.kind != 'f':
        raise ValueError('Ожидается двумерный массив векторов с плавающей точкой.')
    if expected_dimension is not None and vectors.shape[1] != expected_dimension:
        raise ValueError(f'Ожидалось {expected_dimension} признаков в векторе, получено {vectors.shape[1]}.')
    if Path(snapshot).suffix.lower() == '.csv':
        frame = pd.read_csv(snapshot, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    elif Path(snapshot).suffix.lower() == '.pkl':
        frame = pd.read_pickle(snapshot)
    else:
        raise ValueError('Доступны CSV со столбцами Идентификатор и Ответ или доверенный локальный PKL.')
    ids = frame['Идентификатор'].fillna('').astype(str).str.strip()
    answers = frame['Ответ'].fillna('').astype(str).str.strip()
    if ids.empty or ids.eq('').any() or ids.duplicated().any() or answers.eq('').any():
        raise ValueError('В снимке кэша пустой ответ или пустой/повторный ID.')
    if vectors.shape[0] != len(ids):
        raise ValueError('Число строк в NPY и снимке разное.')
    matches = [(i, corpus.index[k]) for i, (k, a) in enumerate(zip(ids, answers))
               if k in corpus.index and corpus.cards[corpus.index[k]].answer.strip() == a]
    if not matches:
        raise ValueError('Ни одного совпадения ID и ответа между выгрузками и снимком.')
    if require_full_coverage and len(matches) != len(corpus.cards):
        raise ValueError(f'Совпали ID и ответы у {len(matches)} из {len(corpus.cards)} FAQ. '
                         'Для полного отчёта нужны векторы для текущей выгрузки.')
    a, b = np.asarray(matches).T
    selected = vectors[a].astype(np.float32)
    lengths = np.linalg.norm(selected, axis=1)
    if not np.isfinite(lengths).all() or (lengths == 0).any():
        raise ValueError('Кэш содержит нулевые или нечисловые векторы среди совпавших FAQ.')
    selected /= lengths[:, None]
    return selected, dict(zip(b.tolist(), range(len(b))))

In [ ]:
"""Benchmark validation, queue construction, and offline reports for FAQ review."""

from collections import Counter, defaultdict
import csv
from hashlib import sha256
from html import escape
import json
from pathlib import Path
from tempfile import TemporaryDirectory

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter



POSITIVE = {'дубль', 'покрытие'}
REVIEW = {'приоритетная вычитка', 'сверка контекста'}
SEVERITY = {'приоритетная вычитка': 0, 'сверка контекста': 1, 'существенное различие': 2}


def fingerprint(card):
    payload = [card.question, *card.alternatives, card.answer, *card.product_key]
    return sha256(json.dumps(payload, ensure_ascii=False).encode()).hexdigest()


def load_benchmark(path, corpus):
    if path is None:
        return {}, 0
    with Path(path).open(encoding='utf-8-sig', newline='') as stream:
        rows = list(csv.DictReader(stream))
    good, stale = {}, 0
    for row in rows:
        a, b = row['id_a'], row['id_b']
        if (a not in corpus.index or b not in corpus.index
                or fingerprint(corpus.cards[corpus.index[a]]) != row['hash_a']
                or fingerprint(corpus.cards[corpus.index[b]]) != row['hash_b']):
            stale += 1
            continue
        key = tuple(sorted((corpus.index[a], corpus.index[b])))
        if key in good:
            raise ValueError(f'Повторная разметка пары {a} и {b}.')
        good[key] = row
    return good, stale










def exact_report(corpus):
    grouped = defaultdict(list)
    for k, card in zip(corpus.form, corpus.cards):
        grouped[k].append(card)
    rows = []
    for cards in grouped.values():
        if len(cards) < 2:
            continue
        for card in cards:
            rows.append((len(rows) + 1, cards[0].id, len(cards), card.id, card.row['Продукт'],
                         card.row['Подпродукт'] or '', card.question, card.answer))
    return rows


def view_row(r):
    a, b = r['cards']
    same, relation, hard, soft = r['flags']
    audit = r['audit']
    queue = r['queue']
    if not r['sources']:
        queue = 'вне пула соседей'
    if answer_key(a.answer) == answer_key(b.answer):
        queue = 'совпадает после нормализации'
    return (queue, 'один продукт' if same else 'между продуктами или продукт пуст',
            r['answer'], r['question'], float(r['cover_a']), float(r['cover_b']), r['embedding'],
            r['signals'], ' | '.join([*hard, *soft]), audit.get('decision', ''),
            audit.get('comment', ''), audit.get('old_verdict', ''), a.id, b.id,
            a.row['Продукт'], a.row['Подпродукт'] or '', b.row['Продукт'], b.row['Подпродукт'] or '',
            a.question, b.question, a.answer, b.answer, ' | '.join(r['sources']), audit.get('source', ''))


PAIR_HEADER = ('Очередь', 'Сравнение', 'Cosine ответа (TF-IDF)', 'Cosine вопроса (TF-IDF)',
               'Доля A в B', 'Доля B в A', 'Cosine E5', 'Сигналы', 'Что проверить',
               'Решение по вычитанной паре', 'Комментарий', 'Старый вердикт ГигаЧата',
               'ID A', 'ID B', 'Продукт A', 'Подпродукт A', 'Продукт B', 'Подпродукт B',
               'Вопрос A', 'Вопрос B', 'Ответ A', 'Ответ B', 'Источник поиска', 'Источник разметки')
EXACT_HEADER = ('Строка', 'Группа', 'Размер', 'ID FAQ', 'Продукт', 'Подпродукт', 'Вопрос', 'Ответ')
METRIC_HEADER = ('Раздел', 'Метод', 'Размеченных пар', 'Найдено полезных', 'Лишних', 'Пропущено полезных',
                 'Доля полезных среди найденных', 'Доля найденных среди полезных в выборке')


def write_workbook(path, summaries, candidates, differences, audited, exact, metrics, lexical, sweep,
                   *, e5=False, pool_counts=()):
    wb = Workbook()
    wb.remove(wb.active)

    def sheet(name, header, data, widths, score_cols=(), row_height=34):
        ws = wb.create_sheet(name)
        ws.append(header)
        for row in data:
            ws.append(row)
            for cell in ws[ws.max_row]:
                if isinstance(cell.value, str):
                    cell.data_type = 's'
            ws.row_dimensions[ws.max_row].height = row_height
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
        ws.sheet_view.showGridLines = False
        ws.row_dimensions[1].height = 34
        for cell in ws[1]:
            cell.fill = PatternFill('solid', fgColor='233F53')
            cell.font = Font(bold=True, color='FFFFFF')
            cell.alignment = Alignment(vertical='center', wrap_text=True)
        for n, width in enumerate(widths, 1):
            ws.column_dimensions[get_column_letter(n)].width = width
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if cell.column in score_cols:
                    cell.number_format = '0.000'
                cell.alignment = Alignment(vertical='top', wrap_text=True)
        return ws

    sheet('Сводка', ('Показатель', 'Значение'), summaries, (43, 98))
    sizes = (23, 23, 18, 18, 13, 13, 13, 32, 52, 20, 50, 53,
             34, 34, 24, 24, 24, 24, 47, 47, 65, 65, 26, 24)
    columns = [i for i in range(len(PAIR_HEADER)) if not e5 or i not in (2, 3)]
    header = tuple(PAIR_HEADER[i] for i in columns)
    widths = tuple(sizes[i] for i in columns)
    score_cols = tuple(n for n, i in enumerate(columns, 1) if 2 <= i <= 6)
    for name, rows, height in (('Неполные дубли', candidates, 70),
                               ('Существенные различия', differences, 58),
                               ('Аудит и разметка', audited, 68)):
        values = (tuple(row[i] for i in columns) for row in map(view_row, rows))
        sheet(name, header, values, widths, score_cols, height)
    sheet('Полные совпадения', EXACT_HEADER, exact, (12, 35, 12, 35, 24, 25, 53, 86), (), 46)
    sheet('Проверка порогов', METRIC_HEADER, metrics, (18, 39, 16, 18, 14, 21, 24, 24), (7, 8))
    if lexical:
        sheet('Сетка TF-IDF', ('Порог ответа', 'Порог вопроса', 'Порог покрытия', *METRIC_HEADER[2:]),
              lexical, (16, 16, 17, 16, 18, 14, 21, 24, 24), (1, 2, 3, 8, 9))
    if sweep:
        sheet('Пороги E5', ('Порог E5', *METRIC_HEADER), sweep,
              (14, 18, 39, 16, 18, 14, 21, 24, 24), (1, 8, 9))
    if pool_counts:
        sheet('Объём по порогам', ('Порог E5', 'Внутри продукта', 'Между продуктами или продукт пуст'),
              pool_counts, (15, 21, 36), (1,))
    wb.save(path)


def write_html(path, summaries, candidates, differences, exact, metrics, *, e5=False):
    def esc(x):
        return escape(str(x), quote=True)

    def pair_table(rows):
        score_header = 'Cosine E5' if e5 else 'Cosine ответа / вопроса'
        out = ['<table><thead><tr><th>Пара и ответы</th><th>Очередь</th>'
               f'<th>{score_header}</th><th>Покрытие A / B</th><th>Замечания</th></tr></thead><tbody>']
        for r in rows:
            a, b = r['cards']
            same, _, hard, soft = r['flags']
            audit = r['audit']
            ids = esc(a.id + ' · ' + b.id)
            product = esc(' / '.join((a.row['Продукт'] or '—', b.row['Продукт'] or '—')))
            notes = ' · '.join(filter(None, [*hard, *soft, audit.get('comment', '')]))
            cosine = f'{r["embedding"]:.3f}' if e5 else f'{r["answer"]:.3f} / {r["question"]:.3f}'
            if not e5 and r['embedding'] is not None:
                cosine += f'<br>E5 {r["embedding"]:.3f}'
            info = esc(audit.get('decision', ''))
            out.append(
                f'<tr data-scope="{int(same)}" data-queue="{esc(r["queue"])}"><td><small>{product}</small>'
                f'<details><summary>{esc(a.question)}<br>{esc(b.question)}</summary>'
                f'<p><small>{ids}</small></p><p><b>Ответ A</b><br>{esc(a.answer)}</p>'
                f'<p><b>Ответ B</b><br>{esc(b.answer)}</p></details></td>'
                f'<td>{esc(r["queue"])}<br><small>{info}</small></td>'
                f'<td>{cosine}</td>'
                f'<td>{r["cover_a"]:.3f} / {r["cover_b"]:.3f}</td><td>{esc(notes)}</td></tr>')
        out.append('</tbody></table>')
        return ''.join(out)

    def simple_table(header, rows, attrs=False):
        return ('<table><thead><tr>' + ''.join(f'<th>{esc(x)}</th>' for x in header) +
                '</tr></thead><tbody>' + ''.join(
                    '<tr' + (' data-scope="" data-queue=""' if attrs else '') + '>' + ''.join(
                        f'<td>{esc(x if x is not None else "—")}</td>' for x in row) + '</tr>' for row in rows) +
                '</tbody></table>')

    body = ['<!doctype html><html lang="ru"><meta charset="utf-8"><meta name="viewport" '
            'content="width=device-width, initial-scale=1"><title>Неполные дубли FAQ</title>',
            """<style>
:root{color-scheme:light;--ink:#19334a;--muted:#516175;--line:#d9e2ea;--bg:#f4f7fa;--accent:#067385}
*{box-sizing:border-box}body{max-width:1440px;margin:auto;padding:24px;background:var(--bg);color:var(--ink);
font:15px/1.5 system-ui,sans-serif}h1{font-size:clamp(28px,5vw,44px);margin:0}h2{margin:.6em 0}
nav,.tools{display:flex;flex-wrap:wrap;gap:8px;margin:16px 0}
button,input,select{font:inherit;border:1px solid #aabac8;border-radius:7px;padding:8px 12px;background:white;color:var(--ink)}
button{cursor:pointer}button[aria-pressed=true]{background:var(--accent);color:white;border-color:var(--accent)}
input{flex:1;min-width:18ch}small{color:var(--muted)}
table{width:100%;border-collapse:collapse;background:white}th,td{text-align:left;padding:9px 12px;
vertical-align:top;border-bottom:1px solid var(--line)}th{background:#e4edf4}td:first-child{min-width:16ch}
summary{cursor:pointer;font-weight:650}details p{white-space:pre-wrap;max-height:28em;overflow:auto}section{overflow:auto}
[hidden]{display:none!important}
</style>""",
            '<h1>Неполные дубли FAQ ЕЛЧ</h1>',
            '<p>Кандидаты по cosine E5 для редакторской проверки.</p>' if e5 else
            '<p>Совпадения и покрытие ответов для редакторской проверки.</p>',
            '<nav><button data-tab="candidates" aria-pressed="true">Кандидаты</button>'
            '<button data-tab="differences" aria-pressed="false">Существенные различия</button>'
            '<button data-tab="exact" aria-pressed="false">Полные совпадения</button>'
            '<button data-tab="quality" aria-pressed="false">Проверка порогов</button></nav>',
            '<div class="tools"><input id="search" aria-label="Поиск" placeholder="Поиск по вопросу, ответу, продукту или ID">'
            '<select id="scope" aria-label="Продукт"><option value="">Все продукты</option>'
            '<option value="1">Внутри продукта</option><option value="0">Между продуктами или продукт пуст</option></select>'
            '<select id="queue" aria-label="Очередь"><option value="">Вся очередь</option>'
            '<option>приоритетная вычитка</option><option>сверка контекста</option></select></div>',
            '<p id="shown" aria-live="polite"></p>',
            '<section id="candidates">' + pair_table(candidates) + '</section>',
            '<section id="differences" hidden>' + pair_table(differences) + '</section>',
            '<section id="exact" hidden>' + simple_table(EXACT_HEADER, exact, True) + '</section>',
            '<section id="quality" hidden><h2>Сводка запуска</h2>' + simple_table(('Показатель','Значение'), summaries)
            + '<h2>Вычитанная выборка</h2>' + simple_table(METRIC_HEADER,
                [(*r[:6], *(None if x is None else f'{x:.1%}' for x in r[6:])) for r in metrics]) + '</section>',
            """<script>
const byId = id => document.getElementById(id);
let tab = 'candidates';
function refresh() {
  byId('shown').hidden = tab === 'quality';
  if (tab === 'quality') return;
  const q = byId('search').value.toLocaleLowerCase('ru').trim();
  const scope = byId('scope').value, queue = byId('queue').value;
  let n = 0;
  for (const row of byId(tab).querySelectorAll('tbody tr')) {
    row.hidden = (q && !row.textContent.toLocaleLowerCase('ru').includes(q))
      || (tab !== 'exact' && scope && row.dataset.scope !== scope)
      || (tab === 'candidates' && queue && row.dataset.queue !== queue);
    if (!row.hidden) n++;
  }
  byId('shown').textContent = 'Показано строк: ' + n;
}
for (const button of document.querySelectorAll('[data-tab]')) {
  button.addEventListener('click', () => {
    byId(tab).hidden = true;
    tab = button.dataset.tab;
    byId(tab).hidden = false;
    for (const item of document.querySelectorAll('[data-tab]')) {
      item.setAttribute('aria-pressed', String(item === button));
    }
    byId('search').disabled = tab === 'quality';
    byId('scope').disabled = tab === 'quality' || tab === 'exact';
    byId('queue').disabled = tab !== 'candidates';
    refresh();
  });
}
for (const id of ['search', 'scope', 'queue']) byId(id).addEventListener('input', refresh);
refresh();
</script></html>"""]
    Path(path).write_text(''.join(body), encoding='utf-8')

In [ ]:
"""FAQ duplicate ranking and calibration on cached E5 answer embeddings."""

from collections import Counter
from dataclasses import dataclass
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory



E5_SPLITS = ('подбор', 'проверка', 'контрольный аудит')
E5_GRID = (.75, .80, .85, .88, .90, .92, .94, .96, .98)
E5_BASE = 'Cosine E5'
E5_RULES = 'E5 и проверки · все продукты'
E5_INSIDE = 'E5 и проверки · внутри продукта'
E5_COVERAGE = 'E5, проверки и вложенность · все продукты'
E5_COVERAGE_INSIDE = 'E5, проверки и вложенность · внутри продукта'
METHODS = (E5_BASE, E5_RULES, E5_INSIDE, E5_COVERAGE, E5_COVERAGE_INSIDE)
COVERAGE_MARGIN = .02


@dataclass
class E5Run:
    corpus: Corpus
    dense: tuple
    pool: dict
    scores: list
    top_k: int


def audit_card_key(ident, content_hash):
    return sha256(f'{ident}\n{content_hash}'.encode()).hexdigest()


def match_saved_audit(corpus, manifest):
    by_key = {audit_card_key(c.id, fingerprint(c)): i for i, c in enumerate(corpus.cards)}
    rows = {}
    for number, (a, b, split, decision) in enumerate(manifest, 1):
        if a in by_key and b in by_key:
            key = tuple(sorted((by_key[a], by_key[b])))
            rows[key] = dict(number=number, split=split, decision=decision,
                             source='Сохранённый аудит по отпечаткам')
    return rows, len(manifest) - len(rows)


def containment_support(row):
    return row['literal'] or (max(row['cover_a'], row['cover_b']) >= .93
                              and (row['same_question'] or min(row['cover_a'], row['cover_b']) >= .65))


def effective_cosine(row, method):
    cosine = row['embedding']
    if method in (E5_COVERAGE, E5_COVERAGE_INSIDE) and containment_support(row):
        return min(1., cosine + COVERAGE_MARGIN)
    return cosine


def e5_rows(corpus, keys, pool, benchmark, dense):
    for key, feature in corpus.features(keys, dense):
        cards = tuple(corpus.cards[i] for i in key)
        same_product, relation, hard, caution = risks(*cards)
        sources = sorted(pool.get(key, ()))
        audit = benchmark.get(key, {})
        if hard:
            queue = 'существенное различие'
        elif relation != 'same' or caution:
            queue = 'сверка контекста'
        else:
            queue = 'приоритетная вычитка'
        yield dict(feature, cards=cards, flags=(same_product, relation, hard, caution),
                   source_key=key, audit=audit, sources=sources, queue=queue, signals='Cosine E5')


def e5_metrics(rows, cutoff, splits=E5_SPLITS):
    metrics = []
    for split in splits:
        cases = [r for r in rows if r['audit']['split'] == split
                 and r['audit']['decision'] != 'уточнить' and r['embedding'] is not None]
        if not cases:
            continue
        for method in METHODS:
            tp = fp = fn = 0
            for row in cases:
                same, _, hard, _ = row['flags']
                selected = (bool(row['sources']) and effective_cosine(row, method) >= cutoff
                            and (method == E5_BASE or not hard)
                            and (method not in (E5_INSIDE, E5_COVERAGE_INSIDE) or same))
                positive = row['audit']['decision'] in POSITIVE
                tp += int(selected and positive)
                fp += int(selected and not positive)
                fn += int(not selected and positive)
            metrics.append((split, method, len(cases), tp, fp, fn,
                            tp / (tp + fp) if tp + fp else None,
                            tp / (tp + fn) if tp + fn else None))
    return metrics


def calibrate_e5(rows, same_product_only):
    """Select by F1 using only the tuning split, then report the comparison grid."""
    tuning = [r for r in rows if r['audit']['split'] == 'подбор'
              and r['audit']['decision'] != 'уточнить' and r['embedding'] is not None]
    labels = {r['audit']['decision'] in POSITIVE for r in tuning
              if not same_product_only or r['flags'][0]}
    cutoffs = sorted({effective_cosine(r, method) for r in tuning if r['sources']
                      for method in (E5_BASE, E5_COVERAGE)} | {.90, .94, .95, .96, .97, .98, .99})
    grid = [(cut, *stats) for cut in cutoffs
            for stats in e5_metrics(tuning, cut, splits=('подбор',))]
    base = E5_INSIDE if same_product_only else E5_RULES
    coverage = E5_COVERAGE_INSIDE if same_product_only else E5_COVERAGE
    ranked = [(2 * tp / (2 * tp + fp + fn), tp, -fp, variant == base, cut, variant)
              for cut, _, variant, _, tp, fp, fn, _, _ in grid if variant in (base, coverage) and tp]
    if len(labels) == 2 and ranked:
        _, _, _, _, cut, method = max(ranked)
        return cut, method, grid
    return None, base, grid


def prepare_e5(exports, embeddings, snapshot, *, top_k=12, block=96, expected_dimension=1024,
               require_full_coverage=False):
    missing = [str(path) for path in (*exports, embeddings, snapshot) if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Для запуска E5 нужны файлы: ' + ', '.join(missing))
    records, _ = read_records(exports)
    corpus = Corpus(records, lexical=False)
    try:
        dense = load_embedding_cache(embeddings, snapshot, corpus, expected_dimension=expected_dimension,
                                     require_full_coverage=require_full_coverage)
    except (ValueError, OSError) as exc:
        raise ValueError(f'Не удалось проверить кэш E5 и снимок строк: {exc}') from exc

    pool = corpus.pool(top_k=top_k, block=block, dense=dense, dense_search_floor=-1.)
    pairs = sorted(pool)
    scores = []
    for start in range(0, len(pairs), 1024):
        batch = pairs[start:start + 1024]
        scores.extend(zip(embedding_cosines(batch, dense), batch))
    scores.sort(key=lambda item: (-item[0], item[1]))
    return E5Run(corpus, dense, pool, scores, top_k)


def product_match(corpus, pair):
    a, b = pair
    return bool(corpus.product[a] and corpus.product[a] == corpus.product[b])


def e5_threshold_counts(prepared):
    scopes = [(score, product_match(prepared.corpus, pair)) for score, pair in prepared.scores]
    return [(cut, sum(score >= cut and same for score, same in scopes),
             sum(score >= cut and not same for score, same in scopes)) for cut in E5_GRID]


def preview_by_band(scores, limit):
    edges = (-1., .80, .85, .90, .95, float('inf'))
    bands = [[item for item in scores if low <= item[0] < high]
             for low, high in zip(edges, edges[1:])]
    bands = [band for band in reversed(bands) if band]
    preview = []
    budget = limit
    for index, band in enumerate(bands):
        remaining = len(bands) - index
        size = min(len(band), (budget + remaining - 1) // remaining)
        if size == 1:
            preview.append(band[len(band) // 2])
        elif size > 1:
            preview.extend(band[i * (len(band) - 1) // (size - 1)] for i in range(size))
        budget -= size
    return preview


def evaluate_e5(prepared, *, benchmark_file=None, saved_audit=(), cutoff=None, same_product_only=False):
    if cutoff is not None and not -1 <= cutoff <= 1:
        raise ValueError('Порог cosine E5 должен быть от −1 до 1.')
    corpus, dense, pool = prepared.corpus, prepared.dense, prepared.pool
    benchmark, missed_audit = match_saved_audit(corpus, saved_audit) if saved_audit else ({}, 0)
    external, stale = load_benchmark(benchmark_file, corpus)
    benchmark.update(external)
    eligible_labels = {key: value for key, value in benchmark.items()
                       if corpus.form[key[0]] != corpus.form[key[1]]
                       and all(index in dense[1] for index in key)}
    audited = list(e5_rows(corpus, eligible_labels, pool, benchmark, dense))
    calibrated, policy, grid = calibrate_e5(audited, same_product_only)
    selected_cutoff = cutoff if cutoff is not None else calibrated
    metrics = e5_metrics(audited, selected_cutoff) if selected_cutoff is not None else []
    return dict(benchmark=benchmark, stale=stale, missed_audit=missed_audit, audited=audited,
                cutoff=selected_cutoff, policy=policy, grid=grid, metrics=metrics)


def export_e5(prepared, output_dir, *, benchmark_file=None, saved_audit=(), cutoff=None,
              same_product_only=False, preview_count=100):
    if preview_count < 1:
        raise ValueError('Размер предварительной выборки должен быть положительным.')
    corpus, dense, pool = prepared.corpus, prepared.dense, prepared.pool
    result = evaluate_e5(prepared, benchmark_file=benchmark_file, saved_audit=saved_audit,
                         cutoff=cutoff, same_product_only=same_product_only)
    benchmark, audited = result['benchmark'], result['audited']
    selected_cutoff, policy = result['cutoff'], result['policy']
    metrics, grid = result['metrics'], result['grid']

    coverage_enabled = policy in (E5_COVERAGE, E5_COVERAGE_INSIDE)
    floor = (selected_cutoff - COVERAGE_MARGIN if coverage_enabled and selected_cutoff is not None
             else selected_cutoff)
    scores = [(score, pair) for score, pair in prepared.scores
              if (not same_product_only or product_match(corpus, pair))
              and (floor is None or score >= floor)]
    if selected_cutoff is None:
        scores = preview_by_band(scores, preview_count)
    rows = list(e5_rows(corpus, dict.fromkeys(pair for _, pair in scores), pool, benchmark, dense))
    if coverage_enabled and selected_cutoff is not None:
        rows = [r for r in rows if effective_cosine(r, policy) >= selected_cutoff]
        for row in rows:
            if row['embedding'] < selected_cutoff:
                row['signals'] = 'Cosine E5 + вложенность'
    for row in rows:
        if row['audit'].get('decision') in ('разные', 'конфликт'):
            row['queue'] = 'существенное различие'
    queue_order = {'приоритетная вычитка': 0, 'сверка контекста': 1, 'существенное различие': 2}
    rows.sort(key=lambda r: (queue_order[r['queue']], -r['embedding'], r['source_key']))
    counts = Counter(r['queue'] for r in rows)
    exact = exact_report(corpus)
    status = ('Просмотр по диапазонам для разметки, порог ещё не выбран' if selected_cutoff is None else
              'Задан вручную' if cutoff is not None else 'Подобран по F1 на части «подбор»')
    summary = [
        ('Метод поиска', 'Cosine E5 по сохранённым векторам ответов'),
        ('Активных карточек в выгрузках', len(corpus.cards)),
        ('Карточек с проверенным E5-кэшем', len(dense[1])),
        ('Карточек без совпавшего вектора', len(corpus.cards) - len(dense[1])),
        ('Лимит соседей на карточку и тип продукта', prepared.top_k),
        ('Пар в пуле соседей E5', len(pool)),
        ('Базовый порог cosine E5', selected_cutoff),
        ('Правило отбора', policy),
        ('Статус порога', status),
        ('Приоритетная вычитка', counts['приоритетная вычитка']),
        ('Сверка контекста', counts['сверка контекста']),
        ('Существенные различия', counts['существенное различие']),
        ('Кандидаты между продуктами', sum(not r['flags'][0] and r['queue'] != 'существенное различие' for r in rows)),
        ('Пар разметки с актуальным текстом', len(audited)),
        ('Пар аудита без совпадения отпечатков', result['missed_audit']),
        ('Устаревших пар разметки', result['stale']),
        ('Покрытие A и B', 'Доля общих взвешенных слов, дополнительный признак для проверки пары.'),
        ('Границы оценки', 'Метрики относятся к размеченной выборке и заданному лимиту соседей.'),
    ]
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    candidates = [r for r in rows if r['queue'] != 'существенное различие']
    differences = [r for r in rows if r['queue'] == 'существенное различие']
    with TemporaryDirectory(dir=output) as temp:
        report, book = Path(temp) / 'faq_e5_report.html', Path(temp) / 'faq_e5_review.xlsx'
        write_html(report, summary, candidates, differences, exact, metrics, e5=True)
        write_workbook(book, summary, candidates, differences, audited, exact, metrics, [], grid,
                       e5=True, pool_counts=e5_threshold_counts(prepared))
        report.replace(output / report.name)
        book.replace(output / book.name)
    return summary


def run_e5(exports, embeddings, snapshot, output_dir, *, benchmark_file=None, top_k=12, block=96,
           cutoff=None, same_product_only=False, preview_count=100, expected_dimension=1024,
           require_full_coverage=False):
    prepared = prepare_e5(exports, embeddings, snapshot, top_k=top_k, block=block,
                          expected_dimension=expected_dimension, require_full_coverage=require_full_coverage)
    return export_e5(prepared, output_dir, benchmark_file=benchmark_file, cutoff=cutoff,
                     same_product_only=same_product_only, preview_count=preview_count)

In [ ]:
import json

SAVED_AUDIT = json.loads(AUDIT_FILE.read_text(encoding="utf-8")) if AUDIT_FILE.is_file() else []
if not SAVED_AUDIT and not BENCHMARK_FILE.is_file():
    print("Для автоматического подбора добавьте faq_e5_audit.json рядом с ноутбуком или укажите CSV с разметкой.")

In [ ]:
from time import perf_counter

started = perf_counter()
prepared = prepare_e5(EXPORT_FILES, E5_NPY, E5_SNAPSHOT, top_k=TOP_K,
                      require_full_coverage=REQUIRE_FULL_COVERAGE)
print(f"Карточек с вектором: {len(prepared.dense[1])} из {len(prepared.corpus.cards)}")
print(f"Найдено пар E5: {len(prepared.pool)}")
print(f"Время расчёта соседей: {perf_counter() - started:.1f} с")
print()
print(f"{'Cosine E5':>10} {'Внутри':>10} {'Между*':>10} {'Всего':>10}")
for cut, inside, across in e5_threshold_counts(prepared):
    print(f"{cut:>10.2f} {inside:>10} {across:>10} {inside + across:>10}")
print("* В том числе пары с незаполненным продуктом.")

In [ ]:
# Настройки можно менять и повторно запускать только эту ячейку.
E5_MIN_COSINE = None
SAME_PRODUCT_ONLY = False
WRITE_REPORT = False

options = dict(
    benchmark_file=BENCHMARK_FILE if BENCHMARK_FILE.is_file() else None,
    saved_audit=SAVED_AUDIT, cutoff=E5_MIN_COSINE, same_product_only=SAME_PRODUCT_ONLY,
)
calibration = evaluate_e5(prepared, **options)
print(f"Пар из файла аудита: {len(SAVED_AUDIT)}")
print(f"Пар с совпавшим текстом и векторами для оценки: {len(calibration['audited'])}")
print(f"Пар без совпадения отпечатков: {calibration['missed_audit']}")
if calibration['cutoff'] is None:
    print("Порог не выбран: проверьте, что в части «подбор» есть оба класса разметки.")
else:
    cut = calibration['cutoff']
    policy = calibration['policy']
    print(f"Порог cosine E5: {cut:.6f}")
    print(f"Правило: {policy}")
    print()
    print("На проверочной выборке при выбранном пороге")
    print(f"{'Метод':<47} {'Пар':>5} {'Найдено':>8} {'Лишних':>8} {'Пропуск':>8} {'F1':>7}")
    for split, method, total, tp, fp, fn, _, _ in calibration['metrics']:
        if split == "проверка":
            f1 = 2 * tp / (2 * tp + fp + fn) if (tp + fp + fn) else 0.
            print(f"{method:<47} {total:>5} {tp:>8} {fp:>8} {fn:>8} {f1:>7.3f}")
    if not any(row[0] == "проверка" for row in calibration['metrics']):
        print("Совпавшей разметки для независимой проверки нет.")
    else:
        for _, method, _, tp, fp, fn, _, _ in e5_metrics(calibration['audited'], .95, splits=('проверка',)):
            if method == E5_INSIDE:
                print(f"Порог 0.95 внутри продукта с проверками — найдено {tp}, лишних {fp}, пропущено {fn}")
if WRITE_REPORT:
    summary = export_e5(prepared, OUTPUT_DIR, **options)
    print(f"Отчёты: {OUTPUT_DIR.resolve()}")